# CS6886W Assignment 1 — Driver Notebook (`driver.ipynb`)
## Exploring VGG6 on CIFAR-10 in TensorFlow & Google Colab

- **GitHub Repository**: [https://github.com/Rakesh4ever/iitm-cs6886w-vgg6-cifar10](https://github.com/Rakesh4ever/iitm-cs6886w-vgg6-cifar10)
- **Purpose**: This `driver.ipynb` notebook acts as the interactive launcher for Google Colab (either in the browser or via the VS Code Google Colab extension). It clones the modular `src/*.py` package onto the Colab GPU runtime and executes training (`src/train.py`), visualization (`outputs/baseline_curves.png`), standalone evaluation (`src/evaluate.py`), and W&B hyperparameter sweeps (`src/sweep.py`).

**Instructions for Evaluators / TAs**:
1. Select a **GPU Runtime** (`Runtime -> Change runtime type -> T4 GPU` in Colab, or `Select Kernel -> Colab -> GPU` in VS Code).
2. Run **Steps 1 through 6** sequentially.

In [ ]:
# ==============================================================================
# STEP 1: CLONE THE MODULAR .py PROJECT FROM GITHUB INTO THE COLAB GPU SERVER
# ==============================================================================
# Purpose of imports:
#   - os: Navigate directories cleanly on Google Colab or local machine
#   - tensorflow: Verify TensorFlow version and confirm T4 GPU hardware availability
import os

if os.path.exists("/content"):
    os.chdir("/content")
    !rm -rf /content/iitm-cs6886w-vgg6-cifar10
    !git clone https://github.com/Rakesh4ever/iitm-cs6886w-vgg6-cifar10.git
    os.chdir("/content/iitm-cs6886w-vgg6-cifar10")

print("Current Working Directory:", os.getcwd())

# Install required dependencies
!pip install -q -r requirements.txt

# Verify modular Python files inside src/
!ls -la src/

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ==============================================================================
# STEP 2: LOGIN TO WEIGHTS & BIASES (W&B) [OPTIONAL FOR LOCAL RUNS]
# ==============================================================================
# Purpose of imports:
#   - os: Set WANDB_API_KEY environment variable for child training processes
#   - wandb: Authenticate session with https://api.wandb.ai
import os
import wandb

# Paste your 40-character W&B API key from https://wandb.ai/authorize below:
WANDB_KEY = os.environ.get("WANDB_API_KEY", "YOUR_WANDB_API_KEY")

if WANDB_KEY != "YOUR_WANDB_API_KEY" and len(WANDB_KEY.strip()) >= 32:
    os.environ["WANDB_API_KEY"] = WANDB_KEY.strip()
    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
else:
    print("W&B API Key not set. Training in Step 3 will run locally and still generate all plots and checkpoints.")

In [ ]:
# ==============================================================================
# STEP 3: TRAIN VGG6 WITH BEST CONFIGURATION (src/train.py)
# ==============================================================================
# Executes modular training pipeline (src/train.py -> src/dataset.py, src/model.py, src/evaluate.py)
# Saves:
#   - Best model checkpoint : checkpoints/best_vgg6_cifar10.keras
#   - Loss & Accuracy plot  : outputs/baseline_curves.png
#   - Metrics JSON          : outputs/latest_results.json
import os
if os.path.exists("/content/iitm-cs6886w-vgg6-cifar10"):
    os.chdir("/content/iitm-cs6886w-vgg6-cifar10")

!python -m src.train \
    --activation gelu \
    --optimizer nadam \
    --batch_size 128 \
    --epochs 20 \
    --learning_rate 0.001 \
    --weight_decay 0.0001 \
    --dropout_rate 0.2 \
    --batch_norm True \
    --augment True \
    --seed 42 \
    --use_wandb \
    --wandb_project cs6886w-vgg6-cifar10

In [ ]:
# ==============================================================================
# STEP 4: VISUALIZE THE SAVED LOSS & ACCURACY PLOT GENERATED BY src/train.py
# ==============================================================================
# Purpose of imports:
#   - os: Verify working directory and plot path
#   - IPython.display (Image, display): Render outputs/baseline_curves.png inline
import os
from IPython.display import Image, display

if os.path.exists("/content/iitm-cs6886w-vgg6-cifar10"):
    os.chdir("/content/iitm-cs6886w-vgg6-cifar10")

display(Image(filename="outputs/baseline_curves.png"))

In [ ]:
# ==============================================================================
# STEP 5: RUN STANDALONE EVALUATION ON THE SAVED MODEL (src/evaluate.py)
# ==============================================================================
# Computes Validation Top-1 Accuracy, Test Top-1 Accuracy, and Per-Class Accuracy
import os
if os.path.exists("/content/iitm-cs6886w-vgg6-cifar10"):
    os.chdir("/content/iitm-cs6886w-vgg6-cifar10")

!python -m src.evaluate \
    --model_path checkpoints/best_vgg6_cifar10.keras \
    --batch_size 128 \
    --seed 42

In [ ]:
# ==============================================================================
# STEP 6: RUN AUTOMATED W&B HYPERPARAMETER SWEEP (src/sweep.py)
# ==============================================================================
# Explores activations, optimizers, batch sizes, learning rates, epochs, and batch norm
import os
if os.path.exists("/content/iitm-cs6886w-vgg6-cifar10"):
    os.chdir("/content/iitm-cs6886w-vgg6-cifar10")

!python -m src.sweep --project cs6886w-vgg6-cifar10 --count 20